In [1]:
from custom_functions import ALPACA_load_data
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

In [89]:
data = pl.read_parquet("data/ELENA_beam_steering_simple_optimiser.parquet").with_columns(pl.all().exclude(['Run Number','SC12_coinc*event_clock']).round(3))
print(data)

shape: (133, 31)
┌────────┬─────────────┬────────────┬────────────┬───┬──────────┬───────────┬──────────┬───────────┐
│ Run    ┆ H_offset_re ┆ V_offset_r ┆ H_angle_re ┆ … ┆ QDNE08P  ┆ QDNE08N   ┆ QDNE14P  ┆ QDNE14N   │
│ Number ┆ quested     ┆ equested   ┆ quested    ┆   ┆ ---      ┆ ---       ┆ ---      ┆ ---       │
│ ---    ┆ ---         ┆ ---        ┆ ---        ┆   ┆ f64      ┆ f64       ┆ f64      ┆ f64       │
│ i32    ┆ f64         ┆ f64        ┆ f64        ┆   ┆          ┆           ┆          ┆           │
╞════════╪═════════════╪════════════╪════════════╪═══╪══════════╪═══════════╪══════════╪═══════════╡
│ 492714 ┆ -10.0       ┆ 0.0        ┆ 0.0        ┆ … ┆ 3165.578 ┆ -3165.83  ┆ 4343.586 ┆ -4343.912 │
│ 492715 ┆ -17.914     ┆ -6.369     ┆ 3.832      ┆ … ┆ 3165.58  ┆ -3165.831 ┆ 4343.588 ┆ -4343.915 │
│ 492716 ┆ -3.952      ┆ 0.31       ┆ -0.898     ┆ … ┆ 3165.58  ┆ -3165.831 ┆ 4343.589 ┆ -4343.915 │
│ 492717 ┆ -17.119     ┆ -4.111     ┆ 3.008      ┆ … ┆ 3165.576 ┆ -3165.82

In [90]:
corrector_names= [
    'DHZE05R',
    'DVTE05T',
    'DHZE08R',
    'DVTE08T',
    'DHZE14R',
    'DVTE14T',
    'QFNE09P',
    'QFNE15P',
    'QDNE08P',
    'QDNE14P',
    ]

parameter_names = [
    'H_offset_mm',
    'V_offset_mm',
    'H_angle_mrad',
    'V_angle_mrad',
]

In [91]:
# for col in parameter_names:
#     data:pl.DataFrame = data.with_columns(pl.col(offset).diff(1).alias(f'delta_{offset}'))
data:pl.DataFrame = data.with_columns(pl.col(parameter_names+corrector_names).diff(1).name.prefix('delta_'))
print(data.select(parameter_names + [f'delta_{offset}' for offset in parameter_names]))

shape: (133, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ H_offset_m ┆ V_offset_m ┆ H_angle_mr ┆ V_angle_m ┆ delta_H_o ┆ delta_V_o ┆ delta_H_a ┆ delta_V_a │
│ m          ┆ m          ┆ ad         ┆ rad       ┆ ffset_mm  ┆ ffset_mm  ┆ ngle_mrad ┆ ngle_mrad │
│ ---        ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ f64        ┆ f64        ┆ f64        ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ -10.0      ┆ 0.0        ┆ 0.0        ┆ 0.0       ┆ null      ┆ null      ┆ null      ┆ null      │
│ -17.914    ┆ -6.369     ┆ 3.832      ┆ 2.874     ┆ -7.914    ┆ -6.369    ┆ 3.832     ┆ 2.874     │
│ -3.952     ┆ 0.31       ┆ -0.898     ┆ 6.006     ┆ 13.962    ┆ 6.679     ┆ -4.73     ┆ 3.132     │
│ -17.119    ┆ -4.111     ┆ 3.008      ┆ 6.821     ┆ -13.167   ┆ -4.421    

In [93]:
from polars import corr


print('##DELTAS')
deltas = data.filter(pl.col('delta_V_offset_mm').is_not_nan()).select([f'delta_{param}' for param in parameter_names]).to_numpy()
deltas = np.hstack([deltas,np.ones((deltas.shape[0],1))])
print(str(deltas[:5][:])[:-1])
print('...')
print(str(deltas[-5:][:])[0:].replace('[',' ',1))

print('##CCORRECTORS')
correctors = data.filter(pl.col('delta_V_offset_mm').is_not_nan()).select([f'delta_{name}' for name in corrector_names]).to_numpy()
print(str(correctors[:5][:])[:-1])
print('...')
print(str(correctors[-5:][:])[0:].replace('[',' ',1))


##DELTAS
[[ -7.914  -6.369   3.832   2.874   1.   ]
 [ 13.962   6.679  -4.73    3.132   1.   ]
 [-13.167  -4.421   3.906   0.815   1.   ]
 [ -2.385  -7.913  -0.064  -0.808   1.   ]
 [  7.385  12.597  -7.223  -3.679   1.   ]
...
 [ -3.007 -11.858   0.     -1.03    1.   ]
 [  0.016   0.      0.      0.      1.   ]
 [  0.      0.      0.      1.03    1.   ]
 [  0.      0.      0.      0.      1.   ]
 [  0.      0.      0.      0.      1.   ]]
##CCORRECTORS
[[ 1.000000e-03  0.000000e+00  1.034095e+03 -6.880340e+02 -2.472350e+02
  -7.305230e+02  3.000000e-03  2.000000e-03  2.000000e-03  2.000000e-03]
 [ 0.000000e+00  1.000000e-03 -1.576403e+03 -6.909610e+02  1.425420e+02
   7.325280e+02  0.000000e+00  0.000000e+00  0.000000e+00  1.000000e-03]
 [-1.000000e-03  0.000000e+00  1.418902e+03 -2.062530e+02 -5.424600e+01
  -4.448000e+00 -4.000000e-03 -5.000000e-03 -4.000000e-03 -7.000000e-03]
 [ 4.000000e-03  2.000000e-03  1.628300e+02 -3.406600e+01  1.018160e+02
  -1.380041e+03  1.000000e-03  2.00

In [94]:
coefficients,residuals,rank,s = np.linalg.lstsq(deltas,correctors,rcond=1e-2)
print(coefficients)
print(residuals)
print(rank)
print(s)

[[ 2.51088544e-05  1.47105465e-05 -6.04429876e+01  9.79180252e-02
  -3.74038083e+01 -1.82541515e+00 -7.17609565e-07  3.84695441e-05
   2.96210175e-05  8.85152915e-05]
 [-8.89052050e-06 -5.28081690e-05  7.74376354e+00  6.19989242e+00
  -3.08753932e-01  1.12672585e+02 -1.87187154e-05  7.58143330e-07
  -2.47862151e-05 -4.74489505e-05]
 [-2.84491849e-05 -5.51701642e-05  1.08470231e+02 -1.81197155e+00
  -1.29587286e+02 -5.20503520e-01 -8.53798141e-05 -8.44250459e-05
  -5.24767079e-05 -9.97074023e-05]
 [-2.54889751e-05 -7.56264979e-05 -2.71389201e+01 -2.32546287e+02
  -2.12459510e+01 -1.27611395e+00  1.05434887e-04 -2.13706490e-05
  -2.72031322e-05 -7.61039818e-05]
 [-2.24483943e-05 -1.52603481e-05  2.45126938e+00  4.44130579e-01
   5.60697745e+00 -8.74050577e-02 -5.65850993e-04 -4.89887655e-04
  -5.84086351e-04 -6.38066432e-04]]
[3.17034338e-04 5.89206324e-04 7.51719453e+06 2.89728954e+05
 8.39447752e+06 1.22001285e+06 8.89328746e-04 7.39099892e-04
 9.04492990e-04 1.16177015e-03]
5
[60.8621

In [101]:
initial_state = data.head(1).select(corrector_names).to_numpy()[0]
print(initial_state)

print(data.select(pl.col('Run Number').min().alias('Run Number min'),pl.col('Run Number').max().alias('Run Number max')))
run_number = 492751
parameters = data.filter(pl.col("Run Number")==492751).select([f'delta_{name}' for name in parameter_names]).to_numpy()[0]
print(parameters)

actual_state = data.filter(pl.col("Run Number")==492751).select([f'delta_{name}' for name in corrector_names]).to_numpy()[0]
print(actual_state)


# print(coefficients.shape)
# print(coefficients[1:][:].shape)
calculated_state = residuals + parameters @ coefficients[1:]
print(calculated_state)
print(actual_state-calculated_state)

[ 150.047 -999.984  345.959 1469.647 -316.811 1436.057 3042.649 1700.898
 3165.578 4343.586]
shape: (1, 2)
┌────────────────┬────────────────┐
│ Run Number min ┆ Run Number max │
│ ---            ┆ ---            │
│ i32            ┆ i32            │
╞════════════════╪════════════════╡
│ 492714         ┆ 492846         │
└────────────────┴────────────────┘
[6.293 0.    0.    1.881]
[ 0.00000e+00  0.00000e+00 -4.50208e+02 -3.63588e+02 -2.44128e+02
 -1.02690e+01  0.00000e+00  1.00000e-03  2.00000e-03  1.00000e-03]
[ 2.18860863e-04  2.28179802e-04  7.51724787e+06  2.89768806e+05
  8.39448613e+06  1.22072173e+06 -2.92833847e-04 -1.77607791e-04
 -3.50153088e-04 -3.37029053e-04]
[-2.18860863e-04 -2.28179802e-04 -7.51769808e+06 -2.90132394e+05
 -8.39473025e+06 -1.22073200e+06  2.92833847e-04  1.17760779e-03
  2.35015309e-03  1.33702905e-03]
